# Day 3 - Bonus: Feature Selection and Pipelines

> **This notebook is optional homework.** Work through it in your own time. There is no separate solution file: every step is worked here, but each one is preceded by a **Try it yourself** prompt. Attempt the prompt first, then run the cell below it to check your thinking.

By the end you will be able to:

- score features three different ways, and explain why the methods disagree
- spot two features that are telling you the same thing
- write one small reusable function and call it instead of repeating yourself
- wrap selection and scaling into a single scikit-learn `Pipeline` that travels with the model

Everything here uses the same feature table you built on Day 3, so the numbers you see are the numbers from your own week.

## 1. Setup

These packages are already in `requirements.txt`. If an import fails, activate `.venv` and run `pip install -r requirements.txt`, then restart the kernel.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pd.set_option('display.float_format', lambda v: f'{v:.4f}')

Load the Day 3 feature table. The loader walks up the folder tree to find `data/`, so this runs from wherever the notebook sits.

In [ ]:
from pathlib import Path

def find_features():
    here = Path.cwd()
    for folder in [here, *here.parents]:
        candidate = folder / 'data' / 'processed' / 'service_requests_features.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find data/processed/service_requests_features.csv')

df = pd.read_csv(find_features())
print('Loaded:', df.shape)
df.head()

### Prepare X and y

On Day 3 you kept only the requests with a known outcome, dropped `resolution_hours` because it leaks the answer, and filled the missing satisfaction scores with the median. Do the same here so this notebook agrees with Day 3.

In [ ]:
HONEST = ['submitted_hour', 'submitted_dow', 'submitted_month', 'is_weekend',
          'is_digital', 'population', 'priority_rank', 'target_resolution_hours',
          'satisfaction_score']

d = df.dropna(subset=['sla_met']).copy()
d['satisfaction_score'] = d['satisfaction_score'].fillna(d['satisfaction_score'].median())

X = d[HONEST]
y = d['sla_met'].astype(int)
print(f'{len(d):,} rows, {X.shape[1]} candidate features')

## 2. Why select features at all?

You have nine features. More is not always better. Some features carry no signal, some repeat what another already says, and every extra feature is one more thing to collect, clean and explain. A smaller model that scores the same is the better model: it is cheaper to run, easier to defend, and less likely to break when the data shifts.

There are three families of method, and this notebook tries one of each:

| Family | Question it asks | Cost |
|---|---|---|
| **Filter** | Which features look related to the target, one at a time? | cheap |
| **Wrapper** | Which subset makes *this model* score best? | expensive |
| **Embedded** | Which features did the model lean on while training? | free, comes with the model |

## 3. Filter methods: score each feature on its own

A filter looks at each feature against the target, independently of any model. Two common scores: the **ANOVA F-test**, which asks whether the feature's average differs between the SLA-met and SLA-missed groups, and **mutual information**, which also catches non-straight-line relationships.

**Try it yourself.** Before running the next cell, guess: which feature will score highest? You met `priority_rank` and `submitted_hour` on Day 3. Write your guess down.

In [ ]:
f_scores = SelectKBest(f_classif, k='all').fit(X, y).scores_
mi_scores = mutual_info_classif(X, y, random_state=42)

filter_table = (pd.DataFrame({'feature': HONEST,
                              'F_score': f_scores,
                              'mutual_info': mi_scores})
                  .sort_values('F_score', ascending=False)
                  .reset_index(drop=True))
filter_table

You should see `priority_rank` a long way ahead on the F-score (around 1576), `is_digital` second (around 497), and everything else close to zero. Mutual information agrees on the top two. The filter's verdict is blunt: two features look useful, the other seven barely register.

## 4. Embedded methods: ask the model what it used

A random forest scores how much each feature helped it split the data. This comes free once the model is trained, which is why it is called *embedded*: the selection is baked into the fitting.

**Try it yourself.** The filter said `priority_rank` was king. Do you expect the forest to agree? Run the cell and compare the two rankings carefully.

In [ ]:
forest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X, y)
importances = (pd.Series(forest.feature_importances_, index=HONEST)
                 .sort_values(ascending=False))
importances

Notice the disagreement. The forest puts **`submitted_hour` top**, with `submitted_month` and `population` high, while the filter had those three near the bottom. The filter also loved `priority_rank`, which the forest ranks only third.

Neither method is wrong. They ask different questions. The F-test only sees straight-line, one-at-a-time relationships, so it misses a feature like `submitted_hour` that matters only in combination with others. The forest sees combinations but spreads importance across features that carry similar information. **When two good methods disagree, that disagreement is information: it usually means the signal is weak and spread thin, which is exactly what you found on Day 3.**

## 5. Wrapper methods: let the model choose the subset

A wrapper trains the model repeatedly, removing the weakest feature each round, until only the number you asked for remains. This is **Recursive Feature Elimination (RFE)**. It is the most direct method, because it optimises the actual model, and the most expensive, because it fits the model many times.

**Try it yourself.** If you had to keep only three features, which would you pick from what you have seen so far? Then run the cell.

In [ ]:
X_scaled = StandardScaler().fit_transform(X)

for k in [3, 5]:
    rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=k).fit(X_scaled, y)
    kept = [f for f, keep in zip(HONEST, rfe.support_) if keep]
    print(f'RFE keeps {k}: {kept}')

With three, RFE keeps `is_digital`, `priority_rank` and `target_resolution_hours`. This is a third answer again, overlapping both earlier methods but identical to neither. Three defensible methods, three different shortlists. Real feature selection is a judgement informed by several views, not a single ranking to obey.

## 6. Two features telling the same story

Before trusting any ranking, check whether features are **correlated with each other**. Two features that move together are partly redundant: keeping both adds cost without adding much signal, and it can make a linear model's coefficients unstable.

**Try it yourself.** Two of these nine features are almost the same measurement wearing different clothes. Can you guess which pair before you run the cell?

In [ ]:
corr = X.corr().abs()

pairs = []
for i in range(len(HONEST)):
    for j in range(i + 1, len(HONEST)):
        if corr.iloc[i, j] > 0.3:
            pairs.append((HONEST[i], HONEST[j], corr.iloc[i, j]))

for a, b, c in sorted(pairs, key=lambda t: -t[2]):
    print(f'{a}  <->  {b}:  {c:.3f}')

`submitted_dow` and `is_weekend` correlate at about **0.79**, and of course they do: the weekend flag is *built from* the day of week. Keeping both hands the model the same fact twice. This is the easiest kind of redundancy to catch, because you can explain it in one sentence without any statistics.

## 7. Writing a reusable function

You have now written the same shape of code three times: fit something, pull out a ranking, sort it. When you notice repetition, that is the moment to write a **function**. A good function has a clear name, takes what it needs as arguments, and returns a result you can use elsewhere.

**Try it yourself.** In the scratch cell below, write a function `top_features(X, y, k)` that returns the names of the top `k` features by F-score. Then compare with the worked version.

In [ ]:
# your turn

Worked version:

In [ ]:
def top_features(X, y, k=5):
    """Return the names of the top-k features by ANOVA F-score."""
    selector = SelectKBest(f_classif, k=k).fit(X, y)
    return list(X.columns[selector.get_support()])

# now the selection is one readable line, reusable anywhere
print('Top 3:', top_features(X, y, 3))
print('Top 5:', top_features(X, y, 5))

The logic has not changed. What changed is that the decision now lives in exactly one place. If you later decide to select by mutual information instead, you edit one function and every caller updates with it. This is the same principle as a dbt macro on Day 4: write the rule once, use it everywhere.

## 8. Putting it together with a Pipeline

There is a trap hiding in everything above. When you scale features or select them using the whole dataset, information from your test rows leaks into the choices you make on your training rows. The fix is a scikit-learn **`Pipeline`**: it chains the steps so that scaling and selection are learned on the training data only, automatically, every time the model is trained.

A pipeline also makes the whole preprocessing-plus-model into a single object you can move around as one thing.

**Try it yourself.** Predict what will happen to accuracy as we cut from nine features down to two. Better, worse, or about the same? Then run the cell.

In [ ]:
# The full model as one object: scale -> select k features -> classify
def build_model(k):
    return Pipeline([
        ('scale', StandardScaler()),
        ('select', SelectKBest(f_classif, k=k)),
        ('model', LogisticRegression(max_iter=1000)),
    ])

# 5-fold cross-validation, so every figure is honest
all_nine = cross_val_score(
    Pipeline([('scale', StandardScaler()),
              ('model', LogisticRegression(max_iter=1000))]),
    X, y, cv=5).mean()
print(f'All 9 features: {all_nine:.4f}')

for k in [2, 3, 5, 7]:
    score = cross_val_score(build_model(k), X, y, cv=5).mean()
    print(f'Top {k} features: {score:.4f}')

This is the punchline of the whole notebook. Every row is about **0.679**. Two features score the same as nine, to four decimal places. The other seven features were carrying almost nothing the first two did not already provide.

That is the real payoff of feature selection. Not a higher score, but the *same* score from a model that is smaller, cheaper, faster to explain, and less likely to break. On Day 3 you learned that when the signal is weak, complex models do not help. This is the same lesson from the other side: when the signal is weak, most of your features do not help either.

## 9. Your turn

Answer these in the scratch cell below. There are no answers printed for these: they are for you to reason through, and to discuss with your instructor if you wish.

1. Rebuild the pipeline using `mutual_info_classif` instead of `f_classif` inside `SelectKBest`. (Hint: `SelectKBest(mutual_info_classif, k=...)`.) Do the top two features change?
2. Drop `is_weekend` entirely, since it duplicates `submitted_dow`. Does the nine-feature accuracy move at all? What does that tell you about the value that feature was adding?
3. In one sentence: if two selection methods disagree about which feature matters most, what would you actually do about it before shipping a model?

In [ ]:
# your turn

---

*End of bonus notebook. Nothing here is required for the assessed labs; it is depth for those who want it.*